# GRPO Training — PGABL Generative AI

**Nama**    : Antonius Vincent Jung  
**Teknik**  : GRPO (Group Relative Policy Optimization) dengan `GRPOTrainer` dari TRL + Unsloth  
**Base**    : Model hasil fine-tuning SFT — `antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA`  
**Tujuan**  : Melatih model agar menghasilkan proses berpikir (`<think>...</think>`) sebelum jawaban akhir

---

## Alur Notebook

| Section | Keterangan |
|---------|------------|
| 1 | Instalasi dependency |
| 2 | Import library |
| 3 | Autentikasi Hugging Face |
| 4 | Load model hasil fine-tuning (SFT) |
| 5 | Load & format dataset Alpaca Indonesia |
| 6 | Reward Functions (4 fungsi) |
| 7 | Konfigurasi GRPOTrainer |
| 8 | Training GRPO |
| 9 | Push model ke Hugging Face Hub |
| 10 | Inferensi & Test Case Wajib |

## Section 1 — Instalasi Dependency

Install Unsloth sebagai framework QLoRA yang dioptimasi, TRL untuk `GRPOTrainer`,
serta library tambahan untuk deteksi bahasa dan evaluasi kemiripan teks.

In [1]:
%%capture
# Unsloth — framework QLoRA yang dioptimasi (2x lebih cepat dari standar HF)
!pip install unsloth -q

# TRL >= 0.17 — berisi GRPOTrainer untuk Reinforcement Learning
!pip install "trl>=0.17.0" -q

# Library pendukung training
!pip install peft accelerate bitsandbytes -q

# Library data & autentikasi
!pip install datasets huggingface_hub -q

# Deteksi bahasa (untuk language_reward_func)
!pip install langdetect -q

# Metrik kemiripan teks (untuk correctness_reward)
!pip install rouge-score -q

print('Instalasi selesai.')

## Section 2 — Import Library

Import semua library yang dibutuhkan untuk proses GRPO.

In [2]:
import os, gc, re, sys
import torch
import numpy as np
from datetime import datetime

from datasets import load_dataset
from huggingface_hub import login as hf_login
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOTrainer, GRPOConfig

from langdetect import detect, LangDetectException
from rouge_score import rouge_scorer

# Verifikasi GPU
assert torch.cuda.is_available(), 'GPU tidak tersedia! Aktifkan T4 di Runtime > Change runtime type'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU   : {gpu_name}')
print(f'VRAM  : {vram_gb:.1f} GB')
print(f'Torch : {torch.__version__}')
print(f'Python: {sys.version.split()[0]}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU   : Tesla T4
VRAM  : 15.6 GB
Torch : 2.10.0+cu128
Python: 3.12.13


## Section 3 — Autentikasi Hugging Face

Login ke Hugging Face menggunakan Kaggle Secrets.  
Tambahkan secret `HF_TOKEN` di panel kiri Kaggle (ikon kunci) sebelum menjalankan cell ini.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# --- Hugging Face ---
hf_token = secrets.get_secret('HF_TOKEN')
hf_login(token=hf_token)
print('Login Hugging Face: OK')

# --- Konfigurasi Repositori ---
HF_USERNAME    = 'antonius-vincent'
SFT_MODEL_REPO = f'{HF_USERNAME}/Qwen2.5-3B-Instruct-Indonesian-QLoRA'   # model input (hasil SFT)
GRPO_REPO_ID   = f'{HF_USERNAME}/Qwen2.5-3B-Instruct-Indonesian-GRPO'    # model output (hasil GRPO)

print(f'Model SFT (input) : {SFT_MODEL_REPO}')
print(f'Model GRPO (output): {GRPO_REPO_ID}')

Login Hugging Face: OK
Model SFT (input) : antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
Model GRPO (output): antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO


## Section 4 — Load Model Hasil Fine-tuning

Memuat kembali model yang telah di-push ke Hugging Face pada tahap fine-tuning (SFT).
Model dimuat dalam format 4-bit (QLoRA) untuk menghemat VRAM GPU T4.

> **Catatan:** `MAX_SEQ_LENGTH` diperkecil menjadi 512 dibanding SFT (2048)
> karena GRPO menghasilkan beberapa completion sekaligus sehingga konsumsi VRAM jauh lebih besar.

In [4]:
MAX_SEQ_LENGTH = 512    # Diperkecil untuk mencegah OOM saat GRPO generate multiple completions
DTYPE          = None   # None = auto-detect (float16 di T4)

print(f'Memuat model SFT: {SFT_MODEL_REPO}')
print('Ini mungkin membutuhkan beberapa menit...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SFT_MODEL_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

print(f'\nModel loaded : {model.__class__.__name__}')
print(f'Vocab size   : {tokenizer.vocab_size:,}')
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM terpakai: {vram_used:.2f} GB / {vram_gb:.2f} GB')

Memuat model SFT: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
Ini mungkin membutuhkan beberapa menit...
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Model loaded : Qwen2ForCausalLM
Vocab size   : 151,643
VRAM terpakai: 2.10 GB / 15.64 GB


### Konfigurasi LoRA untuk GRPO

Menambahkan LoRA adapter baru di atas model SFT agar GRPO hanya melatih
parameter berdimensi rendah (rank 8), bukan seluruh bobot model.
`lora_dropout = 0.0` direkomendasikan untuk GRPO agar gradient lebih stabil.

In [5]:
# Definisi target modules — sama dengan SFT agar adapter bisa ditumpuk
LORA_TARGET_MODULES = [
    'q_proj',    # Query projection  (Multi-Head Attention)
    'k_proj',    # Key projection    (Multi-Head Attention)
    'v_proj',    # Value projection  (Multi-Head Attention)
    'o_proj',    # Output projection (Multi-Head Attention)
    'gate_proj', # Gate projection   (Feed Forward Network / SwiGLU)
    'up_proj',   # Up projection     (Feed Forward Network)
    'down_proj', # Down projection   (Feed Forward Network)
]

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 8,
    target_modules             = LORA_TARGET_MODULES,
    lora_alpha                 = 16,
    lora_dropout               = 0.0,    # 0 direkomendasikan untuk GRPO
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameter trainable : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'Parameter total     : {total:,}')

Unsloth 2026.7.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Parameter trainable : 14,966,784 (0.87%)
Parameter total     : 1,713,639,424


## Section 5 — Load & Format Dataset Alpaca Indonesia

Memuat dataset `Ichsan2895/alpaca-gpt4-indonesian` dan memformatnya ke dalam
struktur yang dibutuhkan `GRPOTrainer`:

- Kolom `prompt` : list of messages dalam format chat (system + user)
- Kolom `answer` : ground truth output — digunakan oleh `correctness_reward`

In [6]:
DATASET_NAME = 'Ichsan2895/alpaca-gpt4-indonesian'
MAX_SAMPLES  = 2_000   # Batasi untuk efisiensi GPU T4
SEED         = 42

print(f'Memuat dataset: {DATASET_NAME} ...')
raw_ds = load_dataset(DATASET_NAME, split='train')
print(f'Jumlah sampel asli: {len(raw_ds):,}')

# Hapus kolom index jika ada
if 'Unnamed: 0' in raw_ds.column_names:
    raw_ds = raw_ds.remove_columns(['Unnamed: 0'])
    print('Kolom index (Unnamed: 0) dihapus.')

# Subsample agar GRPO lebih cepat di T4
if len(raw_ds) > MAX_SAMPLES:
    raw_ds = raw_ds.shuffle(seed=SEED).select(range(MAX_SAMPLES))
    print(f'Dataset disubsample ke {MAX_SAMPLES:,} sampel')

print(f'\nKolom dataset  : {raw_ds.column_names}')
print('Contoh pertama:')
print(f'  input  : {raw_ds[0].get("input",  "")[:100]}')
print(f'  output : {raw_ds[0].get("output", "")[:100]}')

Memuat dataset: Ichsan2895/alpaca-gpt4-indonesian ...


README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah sampel asli: 49,969
Kolom index (Unnamed: 0) dihapus.
Dataset disubsample ke 2,000 sampel

Kolom dataset  : ['input', 'output']
Contoh pertama:
  input  : Tentukan hubungan antara variabel-variabel berikut.
Umur dan kecerdasan.
  output : Usia dan kecerdasan adalah dua variabel terpisah yang dapat memiliki hubungan kompleks. Meskipun usi


### Format Dataset untuk GRPOTrainer

System prompt dirancang untuk mendorong model menggunakan tag `<think>...</think>`
sebelum memberikan jawaban akhir.

In [7]:
SYSTEM_PROMPT = (
    'Kamu adalah asisten AI yang cerdas dan membantu. '
    'Sebelum menjawab, tuliskan proses berpikirmu di dalam tag <think>...</think>, '
    'kemudian berikan jawaban akhir yang jelas dan lengkap dalam Bahasa Indonesia.'
)

def format_for_grpo(example):
    """
    Format satu baris dataset ke dict yang dibutuhkan GRPOTrainer.

    Kolom dataset: 'input' -> pesan user, 'output' -> respons asisten

    Returns:
        dict berisi 'prompt' (list of messages) dan 'answer' (ground truth)
    """
    user_msg = example.get('input',  '').strip()
    answer   = example.get('output', '').strip()

    if not user_msg or not answer:
        return {'prompt': None, 'answer': None}

    prompt = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]
    return {'prompt': prompt, 'answer': answer}


print('Memformat dataset untuk GRPOTrainer...')
grpo_ds = raw_ds.map(
    format_for_grpo,
    remove_columns=raw_ds.column_names,
    batched=False,
    desc='Format GRPO',
)

# Hapus baris kosong
grpo_ds = grpo_ds.filter(lambda x: x['prompt'] is not None and x['answer'] is not None)

print(f'Selesai  : {len(grpo_ds):,} sampel siap')
print(f'Kolom    : {grpo_ds.column_names}')
print('\nContoh prompt[0]:')
for msg in grpo_ds[0]['prompt']:
    print(f'  [{msg["role"]}] {msg["content"][:90]}...')
print(f'  [answer] {grpo_ds[0]["answer"][:90]}...')

Memformat dataset untuk GRPOTrainer...


Format GRPO:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Selesai  : 2,000 sampel siap
Kolom    : ['prompt', 'answer']

Contoh prompt[0]:
  [system] Kamu adalah asisten AI yang cerdas dan membantu. Sebelum menjawab, tuliskan proses berpiki...
  [user] Tentukan hubungan antara variabel-variabel berikut.
Umur dan kecerdasan....
  [answer] Usia dan kecerdasan adalah dua variabel terpisah yang dapat memiliki hubungan kompleks. Me...


## Section 6 — Reward Functions

Empat reward function yang mendorong model menghasilkan output berkualitas:

| Fungsi | Tujuan | Skor Maks |
|--------|--------|-----------|
| `format_reward_func` | Format `<think>...</think>` yang benar | +1.0 |
| `reasoning_length_reward` | Proses berpikir yang detail | +1.0 |
| `correctness_reward` | Jawaban sesuai ground truth | +1.0 |
| `language_reward_func` | Menjawab dalam Bahasa Indonesia | +1.0 |

Total skor maksimum per completion: **+4.0**

### 6.1 — `format_reward_func`

Menggunakan strategi **Reward Shaping** dengan poin bertahap agar model belajar
format secara progresif, dari membuka tag hingga format sempurna.

| Kondisi | Poin |
|---------|------|
| Model membuka respons dengan tag `<think>` | +0.2 |
| Model menutup pemikirannya dengan tag `</think>` | +0.3 |
| Format sempurna: di awal, tertutup, diikuti jawaban akhir | +1.0 |
| **Penalti**: tag `<think>` atau `</think>` muncul lebih dari satu kali | −0.5 |

In [8]:
def format_reward_func(completions, **kwargs):
    """
    Reward shaping bertahap untuk format <think>...</think>.

    Poin bertahap:
      +0.2  jika model membuka dengan <think>
      +0.3  jika model menutup dengan </think>
      +1.0  jika format sempurna (di awal, tertutup, diikuti jawaban akhir)
      -0.5  penalti jika <think> atau </think> muncul lebih dari satu kali

    Args:
        completions : list of str — output model per-sample
    Returns:
        list of float
    """
    rewards = []

    for completion in completions:
        # GRPOTrainer mengirim completions sebagai string
        text = completion if isinstance(completion, str) else str(completion)

        score = 0.0
        open_count  = text.count('<think>')
        close_count = text.count('</think>')

        # Penalti halusinasi: tag muncul lebih dari satu kali
        if open_count > 1 or close_count > 1:
            score = -0.5
        else:
            if open_count == 1:     # Ada <think>
                score += 0.2
            if close_count == 1:    # Ada </think>
                score += 0.3

            # Format sempurna: dimulai <think>, tertutup </think>, ada jawaban sesudahnya
            perfect = re.compile(
                r'^\s*<think>.+?</think>\s*.+',
                re.DOTALL
            )
            if perfect.match(text):
                score = 1.0   # Override ke poin maksimum

        rewards.append(score)
    return rewards


# --- Unit test ---
_fmt_tests = [
    '<think>Saya perlu cek UU ketenagakerjaan.</think> Ya, Anda berhak.',  # sempurna
    '<think>Berpikir sebentar.',                                            # tidak tertutup
    'Jawaban langsung tanpa think.',                                        # tidak ada tag
    '<think>A</think><think>B</think> Jawaban.',                           # halusinasi
]
print('=== Test format_reward_func ===')
for tc in _fmt_tests:
    s = format_reward_func([tc])
    print(f'  Score={s[0]:+.1f} | {tc[:65]}')

=== Test format_reward_func ===
  Score=+1.0 | <think>Saya perlu cek UU ketenagakerjaan.</think> Ya, Anda berhak
  Score=+0.2 | <think>Berpikir sebentar.
  Score=+0.0 | Jawaban langsung tanpa think.
  Score=-0.5 | <think>A</think><think>B</think> Jawaban.


### 6.2 — `reasoning_length_reward`

Memberikan poin proporsional berdasarkan **panjang karakter** teks di dalam
`<think>...</think>`. Fungsi ini toleran jika `</think>` terpotong batas token
(tetap menghitung teks setelah `<think>` sebagai konten berpikir).

| Kondisi | Poin |
|---------|------|
| Tidak ada `<think>`, atau isinya kosong/hanya spasi | +0.0 |
| Panjang isi `<think>` < 50 karakter (terlalu singkat) | +0.2 |
| Panjang isi `<think>` antara 50–199 karakter | +0.5 |
| Panjang isi `<think>` ≥ 200 karakter (penalaran detail) | +1.0 |

In [9]:
def reasoning_length_reward(completions, **kwargs):
    """
    Reward proporsional berdasarkan panjang konten di dalam <think>...</think>.
    Toleran jika </think> terpotong batas token maksimum.

    Args:
        completions : list of str
    Returns:
        list of float
    """
    rewards = []

    for completion in completions:
        text = completion if isinstance(completion, str) else str(completion)

        open_idx = text.find('<think>')
        if open_idx == -1:
            rewards.append(0.0)   # Tidak ada tag <think>
            continue

        content_start = open_idx + len('<think>')
        close_idx     = text.find('</think>', content_start)

        if close_idx != -1:
            think_content = text[content_start:close_idx]
        else:
            think_content = text[content_start:]  # Terpotong: ambil sisa teks

        think_len = len(think_content.strip())

        if think_len == 0:
            score = 0.0    # Kosong atau hanya spasi
        elif think_len < 50:
            score = 0.2    # Terlalu singkat
        elif think_len < 200:
            score = 0.5    # Sedang (50–199 karakter)
        else:
            score = 1.0    # Penalaran detail (>= 200 karakter)

        rewards.append(score)
    return rewards


# --- Unit test ---
_len_tests = [
    '<think></think> Jawaban.',
    '<think>Ok.</think> Jawaban.',
    '<think>' + 'A' * 80  + '</think> Jawaban.',
    '<think>' + 'A' * 200 + '</think> Jawaban.',
    '<think>' + 'A' * 150,                          # terpotong
]
print('=== Test reasoning_length_reward ===')
for tc in _len_tests:
    s   = reasoning_length_reward([tc])
    start = tc.find('<think>') + 7
    end   = tc.find('</think>') if '</think>' in tc else len(tc)
    n   = len(tc[start:end].strip())
    print(f'  Score={s[0]:+.1f} | think_len={n:>4} | {tc[:55]}...')

=== Test reasoning_length_reward ===
  Score=+0.0 | think_len=   0 | <think></think> Jawaban....
  Score=+0.2 | think_len=   3 | <think>Ok.</think> Jawaban....
  Score=+0.5 | think_len=  80 | <think>AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
  Score=+1.0 | think_len= 200 | <think>AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
  Score=+0.5 | think_len= 150 | <think>AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...


### 6.3 — `correctness_reward`

Membandingkan jawaban akhir model (teks setelah `</think>`) dengan
ground truth dari kolom `output` dataset menggunakan metrik **ROUGE-L**.

| Kondisi | Poin |
|---------|------|
| ROUGE-L score ≥ 0.3 (jawaban relevan dengan ground truth) | +1.0 |
| ROUGE-L score < 0.3 | +0.0 |

In [10]:
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def correctness_reward(completions, answer, **kwargs):
    """
    Reward +1.0 jika jawaban akhir model memiliki kemiripan ROUGE-L >= threshold
    dengan ground truth (kolom 'answer' dari dataset).

    Args:
        completions : list of str — output model
        answer      : list of str — ground truth dari dataset
    Returns:
        list of float
    """
    THRESHOLD = 0.3
    rewards   = []

    for completion, gt in zip(completions, answer):
        text = completion if isinstance(completion, str) else str(completion)

        # Ambil teks jawaban akhir (setelah </think> jika ada)
        close_idx = text.find('</think>')
        if close_idx != -1:
            final_answer = text[close_idx + len('</think>'):].strip()
        else:
            final_answer = text.strip()

        if not final_answer or not gt:
            rewards.append(0.0)
            continue

        # Hitung ROUGE-L
        result  = _rouge.score(gt, final_answer)
        rouge_l = result['rougeL'].fmeasure

        rewards.append(1.0 if rouge_l >= THRESHOLD else 0.0)

    return rewards


# --- Unit test ---
_corr_comps   = [
    '<think>Cek UU Ketenagakerjaan.</think> Ya, Anda berhak mendapat uang lembur berdasarkan UU.',
    '<think>Tidak relevan.</think> Ini tentang topik yang sama sekali berbeda.',
]
_corr_answers = [
    'Anda berhak mendapat uang lembur sesuai UU Ketenagakerjaan.',
    'Anda berhak mendapat uang lembur sesuai UU Ketenagakerjaan.',
]
print('=== Test correctness_reward ===')
scores = correctness_reward(_corr_comps, _corr_answers)
for i, (s, c) in enumerate(zip(scores, _corr_comps)):
    print(f'  Score={s:+.1f} | {c[:65]}')

=== Test correctness_reward ===
  Score=+1.0 | <think>Cek UU Ketenagakerjaan.</think> Ya, Anda berhak mendapat u
  Score=+0.0 | <think>Tidak relevan.</think> Ini tentang topik yang sama sekali 


### 6.4 — `language_reward_func`

Mengevaluasi bahasa yang digunakan pada jawaban akhir model (teks setelah `</think>`).
Hanya teks jawaban akhir yang dievaluasi, bukan proses berpikir.

| Kondisi | Poin |
|---------|------|
| Jawaban akhir dalam **Bahasa Indonesia** | +1.0 |
| Jawaban akhir dalam **Bahasa Inggris** | −0.5 |
| Bahasa lain atau terlalu pendek untuk dideteksi | +0.0 |

In [11]:
def language_reward_func(completions, **kwargs):
    """
    Reward berdasarkan bahasa jawaban akhir model (setelah </think>).

      +1.0  — Bahasa Indonesia
      -0.5  — Bahasa Inggris (penalti)
       0.0  — Bahasa lain atau tidak terdeteksi

    Args:
        completions : list of str
    Returns:
        list of float
    """
    rewards = []

    for completion in completions:
        text = completion if isinstance(completion, str) else str(completion)

        # Ambil teks jawaban akhir (setelah </think>)
        close_idx = text.find('</think>')
        if close_idx != -1:
            final_text = text[close_idx + len('</think>'):].strip()
        else:
            final_text = text.strip()

        # Terlalu pendek untuk deteksi bahasa yang andal
        if len(final_text) < 10:
            rewards.append(0.0)
            continue

        try:
            lang = detect(final_text)
            if lang == 'id':
                rewards.append(1.0)    # Bahasa Indonesia — reward
            elif lang == 'en':
                rewards.append(-0.5)   # Bahasa Inggris — penalti
            else:
                rewards.append(0.0)
        except LangDetectException:
            rewards.append(0.0)

    return rewards


# --- Unit test ---
_lang_tests = [
    '<think>Berpikir...</think> Ya, Anda berhak mendapat uang lembur.',
    '<think>Thinking...</think> Yes, you are entitled to overtime pay.',
    '<think>Berpikir...</think>',
]
print('=== Test language_reward_func ===')
for tc in _lang_tests:
    s = language_reward_func([tc])
    print(f'  Score={s[0]:+.1f} | {tc[:65]}')

print('\nSemua reward functions siap.')

=== Test language_reward_func ===
  Score=+1.0 | <think>Berpikir...</think> Ya, Anda berhak mendapat uang lembur.
  Score=-0.5 | <think>Thinking...</think> Yes, you are entitled to overtime pay.
  Score=+0.0 | <think>Berpikir...</think>

Semua reward functions siap.


## Section 7 — Konfigurasi GRPOTrainer

Mengatur hyperparameter GRPO yang aman untuk GPU T4 (15 GB VRAM).

Parameter kritis untuk mencegah OOM:

| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `num_generations` | 4 | Jumlah completion per prompt (default 8 → dikecilkan) |
| `max_completion_length` | 256 | Panjang maks output yang digenerate |
| `max_prompt_length` | 256 | Panjang maks prompt yang masuk |
| `per_device_train_batch_size` | 1 | Harus 1 di T4 untuk GRPO |

In [12]:
grpo_config = GRPOConfig(
    # --- Output ---
    output_dir                  = './grpo_output',

    # --- Generasi (kritis untuk T4 — dikecilkan agar tidak OOM) ---
    num_generations             = 4,      # Jumlah completion per prompt (default: 8)
    max_completion_length       = 256,    # Panjang maks output yang digenerate
    max_prompt_length           = 256,    # Panjang maks prompt

    # --- Training ---
    per_device_train_batch_size = 1,      # Harus 1 di T4 untuk GRPO
    gradient_accumulation_steps = 4,      # Effective batch = 1 * 4 = 4
    max_steps                   = 200,    # Steps training (naikkan untuk hasil lebih baik)
    num_train_epochs            = 1,

    # --- Optimizer & Learning Rate ---
    learning_rate               = 5e-6,   # LR kecil untuk GRPO (hindari policy collapse)
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,

    # --- Presisi ---
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),

    # --- KL Divergence ---
    # Mengontrol seberapa jauh model boleh menyimpang dari SFT baseline
    beta                     = 0.1,

    # --- Logging ---
    logging_steps               = 10,
    report_to                   = 'none',
    seed                        = 42,
)

print('Konfigurasi GRPOTrainer:')
print(f'  num_generations        : {grpo_config.num_generations}')
print(f'  max_completion_length  : {grpo_config.max_completion_length}')
print(f'  max_prompt_length      : {grpo_config.max_prompt_length}')
print(f'  per_device_train_batch : {grpo_config.per_device_train_batch_size}')
print(f'  gradient_accum_steps   : {grpo_config.gradient_accumulation_steps}')
print(f'  max_steps              : {grpo_config.max_steps}')
print(f'  learning_rate          : {grpo_config.learning_rate}')
print(f'  beta (kl_coef)         : {grpo_config.beta}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Konfigurasi GRPOTrainer:
  num_generations        : 4
  max_completion_length  : 256
  max_prompt_length      : 256
  per_device_train_batch : 1
  gradient_accum_steps   : 4
  max_steps              : 200
  learning_rate          : 5e-06
  beta (kl_coef)         : 0.1


## Section 8 — Training GRPO

Menjalankan `GRPOTrainer` dengan empat reward functions sekaligus.

**Cara kerja GRPO:**
1. Untuk setiap prompt, model menghasilkan `num_generations` completion
2. Setiap completion dinilai oleh semua reward functions (skor dijumlahkan)
3. Completion dengan skor lebih tinggi diberi bobot lebih besar dalam update
4. KL divergence term menjaga model agar tidak terlalu menyimpang dari SFT baseline

In [13]:
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args             = grpo_config,
    train_dataset    = grpo_ds,
)

print('GRPOTrainer berhasil diinisialisasi.')
print(f'Total sampel training  : {len(grpo_ds):,}')
print(f'Reward functions aktif : {len(trainer.reward_funcs)}')

vram_before = torch.cuda.memory_allocated() / 1e9
print(f'VRAM sebelum training  : {vram_before:.2f} GB / {vram_gb:.2f} GB')

GRPOTrainer berhasil diinisialisasi.
Total sampel training  : 2,000
Reward functions aktif : 4
VRAM sebelum training  : 2.16 GB / 15.64 GB


In [14]:
# --- Mulai Training ---
print(f'Training dimulai: {datetime.now().strftime("%H:%M:%S")}')
print(f'Max steps        : {grpo_config.max_steps}')
print(f'Num generations  : {grpo_config.num_generations}')
print()

train_result = trainer.train()

print()
print(f'Training selesai: {datetime.now().strftime("%H:%M:%S")}')
print(f'Training loss   : {train_result.training_loss:.4f}')

Training dimulai: 19:50:02
Max steps        : 200
Num generations  : 4



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` w

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
10,0.000001,1.385000,0.276470,178.625000,135.900000,224.500000,0.450000,101.308334,84.700000,131.700000,0.000014,0.085000,0.097258,0.175000,0.180606,0.225000,0.207735,0.900000,0.070711
20,0.000002,0.967500,0.135000,155.175000,136.000000,175.300000,0.400000,52.775000,33.600000,72.900000,0.000017,0.005000,0.010000,0.025000,0.050000,0.000000,0.000000,0.937500,0.075000
30,0.000002,1.100000,0.157735,204.750000,171.300000,231.000000,0.600000,65.733334,43.300000,87.500000,0.000016,0.000000,0.000000,0.000000,0.000000,0.100000,0.157735,1.000000,0.000000
40,0.000002,1.412500,0.454748,136.225000,74.500000,194.500000,0.175000,125.566667,74.500000,180.200000,0.000022,0.075000,0.128868,0.137500,0.232735,0.200000,0.257735,1.000000,0.000000
50,0.000027,1.217500,0.445226,127.975000,92.000000,169.700000,0.275000,72.175000,40.800000,112.500000,0.000267,0.042500,0.073629,0.087500,0.147871,0.225000,0.150000,0.862500,0.120650
60,0.000091,1.067500,0.278145,126.250000,95.900000,161.000000,0.250000,86.450000,70.300000,109.600000,0.000909,0.017500,0.035000,0.050000,0.100000,0.100000,0.100000,0.900000,0.100000
70,0.000224,1.325000,0.421819,182.800000,137.400000,221.300000,0.400000,117.166669,86.200000,149.500000,0.002238,0.050000,0.080176,0.175000,0.265470,0.100000,0.115470,1.000000,0.000000
80,0.000632,1.440000,0.457168,176.150000,161.400000,201.300000,0.525000,71.175000,59.000000,94.500000,0.006321,0.097500,0.110616,0.355000,0.310000,0.050000,0.100000,0.937500,0.125000
90,0.000625,1.427500,0.404065,173.300000,140.800000,227.500000,0.475000,83.658333,64.000000,121.500000,0.006249,0.065000,0.084483,0.162500,0.205606,0.225000,0.250000,0.975000,0.050000
100,0.000699,1.585000,0.584196,199.000000,159.100000,238.500000,0.550000,79.041667,56.700000,104.000000,0.006994,0.147500,0.168384,0.462500,0.413675,0.025000,0.050000,0.950000,0.100000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformer


Training selesai: 21:22:05
Training loss   : 0.0009


## Section 9 — Push Model Hasil GRPO ke Hugging Face Hub

Merge LoRA adapter GRPO ke base model dan push dalam format `merged_16bit`
agar model dapat digunakan langsung tanpa library PEFT.

In [15]:
print(f'Push model GRPO ke: {GRPO_REPO_ID}')
print('Format: merged_16bit (LoRA digabung dengan base model dalam float16)')

model.push_to_hub_merged(
    repo_id     = GRPO_REPO_ID,
    tokenizer   = tokenizer,
    save_method = 'merged_16bit',
    token       = hf_token,
)

print()
print('Model berhasil di-push ke Hugging Face!')
print(f'URL: https://huggingface.co/{GRPO_REPO_ID}')

Push model GRPO ke: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO
Format: merged_16bit (LoRA digabung dengan base model dalam float16)


Unsloth: Restored added_tokens_decoder metadata in antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...


Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:06<01:06, 66.12s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:42<00:00, 51.43s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO`

Model berhasil di-push ke Hugging Face!
URL: https://huggingface.co/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO


### Simpan Link ke `link_huggingface.txt`

Simpan URL kedua model (SFT dan GRPO) ke dalam file `link_huggingface.txt`
sesuai ketentuan submission.

In [16]:
with open('link_huggingface.txt', 'w') as f:
    f.write(f'SFT Model  : https://huggingface.co/{SFT_MODEL_REPO}\n')
    f.write(f'GRPO Model : https://huggingface.co/{GRPO_REPO_ID}\n')

print('link_huggingface.txt diperbarui:')
with open('link_huggingface.txt') as f:
    print(f.read())

link_huggingface.txt diperbarui:
SFT Model  : https://huggingface.co/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
GRPO Model : https://huggingface.co/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-GRPO



## Section 10 — Inferensi dan Test Case Wajib

Menguji model hasil GRPO menggunakan prompt test case yang diwajibkan rubrik.

**Ekspektasi output:**
```
<think>
User adalah staf admin (pekerja non-manajerial). Berdasarkan PP 35/2021,
pekerja yang bekerja melebihi waktu kerja standar wajib dibayar upah lembur...
</think>

Ya, Anda berhak. Berdasarkan PP No. 35 Tahun 2021, perusahaan wajib membayar
upah lembur untuk staf admin yang bekerja melebihi waktu kerja normal (8 jam sehari).
```

In [17]:
# Aktifkan mode inference Unsloth (lebih cepat, matikan gradient)
FastLanguageModel.for_inference(model)

def generate_response(instruction, max_new_tokens=512):
    """
    Generate respons menggunakan model hasil GRPO.
    Model dilatih menghasilkan <think>...</think> sebelum jawaban akhir.
    """
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.6,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# --- Test Case Wajib (dari rubrik Advanced) ---
WAJIB_PROMPT = (
    'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. '
    'Apakah saya berhak dapat uang lembur?'
)

print('=' * 65)
print('TEST CASE WAJIB — RUBRIK ADVANCED')
print('=' * 65)
print(f'Prompt: {WAJIB_PROMPT}')
print()
print('Generating...')
response = generate_response(WAJIB_PROMPT)
print()
print('=' * 65)
print('RESPONS MODEL:')
print('=' * 65)
print(response)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST CASE WAJIB — RUBRIK ADVANCED
Prompt: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Generating...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



RESPONS MODEL:
<think>
Pertama-tama, penting untuk memahami bahwa aturan tentang uang lembur bervariasi dari satu perusahaan ke perusahaan lainnya, tergantung pada budaya kerja, praktik, dan kebijakan internal. Namun, beberapa perusahaan mungkin memiliki kebijakan atau aturan tertentu mengenai uang lembur.

Secara umum, uang lembur biasanya diberikan kepada karyawan ketika mereka bekerja lebih dari waktu kerja yang ditetapkan secara normal. Hal ini dapat termasuk jam kerja tambahan, hari libur, atau waktu istirahat tambahan. Uang lembur sering disebut sebagai "jam kerja tambahan" atau "lembur".

Untuk mengetahui apakah Anda berhak mendapatkan uang lembur, Anda harus memeriksa aturan dan kebijakan internal perusahaan Anda. Jika ada, sebaiknya Anda mencari informasi lebih lanjut dari manajemen atau HR untuk memastikan bahwa Anda benar-benar berhak mendapatkannya.

Selain itu, penting untuk mempertimbangkan apakah pekerjaan tersebut dilakukan dengan baik dan sesuai dengan standar profesi

### Validasi Format Output

Memeriksa apakah output model memenuhi format yang disyaratkan rubrik:
- Terdapat tag `<think>` dan `</think>`
- Ada teks reasoning di dalam tag
- Ada teks jawaban akhir setelah `</think>`

In [18]:
print('=== VALIDASI FORMAT OUTPUT ===')
has_open   = '<think>'  in response
has_close  = '</think>' in response
has_answer = len(response.split('</think>')[-1].strip()) > 10 if has_close else False

print(f'  Ada <think>       : {"✅" if has_open   else "❌"}')
print(f'  Ada </think>      : {"✅" if has_close  else "❌"}')
print(f'  Ada jawaban akhir : {"✅" if has_answer else "❌"}')

if has_open and has_close:
    think_start   = response.find('<think>') + len('<think>')
    think_end     = response.find('</think>')
    think_content = response[think_start:think_end].strip()
    final_answer  = response[response.find('</think>') + len('</think>'):].strip()

    print()
    print(f'  Panjang <think>   : {len(think_content)} karakter')
    print(f'  Panjang jawaban   : {len(final_answer)} karakter')
    print()
    print(f'  [THINK]  {think_content[:150]}{"..." if len(think_content)>150 else ""}')
    print()
    print(f'  [ANSWER] {final_answer[:200]}{"..." if len(final_answer)>200 else ""}')

overall_pass = has_open and has_close and has_answer
print()
print('=' * 65)
status = '✅ LULUS — Format <think> terdeteksi' if overall_pass else '❌ GAGAL — Format tidak sesuai'
print(f'STATUS: {status}')
print('=' * 65)

=== VALIDASI FORMAT OUTPUT ===
  Ada <think>       : ✅
  Ada </think>      : ✅
  Ada jawaban akhir : ✅

  Panjang <think>   : 1077 karakter
  Panjang jawaban   : 246 karakter

  [THINK]  Pertama-tama, penting untuk memahami bahwa aturan tentang uang lembur bervariasi dari satu perusahaan ke perusahaan lainnya, tergantung pada budaya ke...

  [ANSWER] Jadi, untuk menentukan apakah Anda berhak mendapatkan uang lembur, Anda harus memeriksa kebijakan internal perusahaan Anda, memverifikasi apakah pekerjaan Anda dilakukan dengan baik, dan meminta klari...

STATUS: ✅ LULUS — Format <think> terdeteksi


## Section 11 — Pengujian Tambahan

Menguji model dengan beberapa pertanyaan hukum ketenagakerjaan lainnya
untuk memverifikasi konsistensi format `<think>` pada berbagai jenis pertanyaan.

In [19]:
test_cases = [
    'Apakah PKWT (kontrak kerja waktu tertentu) boleh mensyaratkan masa percobaan kerja?',
    'Apa saja hak cuti yang dimiliki pekerja berdasarkan UU Ketenagakerjaan?',
    'Bagaimana ketentuan upah minimum menurut peraturan yang berlaku?',
    'Apa yang dimaksud dengan PHK dan apa saja alasan yang diperbolehkan?',
]

print('=== PENGUJIAN TAMBAHAN ===')
for i, q in enumerate(test_cases, 1):
    print(f'\n{"="*60}')
    print(f'Pertanyaan {i}: {q}')
    print('='*60)
    resp      = generate_response(q, max_new_tokens=384)
    has_think = '<think>' in resp and '</think>' in resp
    print(resp[:400] + ('...' if len(resp) > 400 else ''))
    print(f'  >> Format <think>: {"✅" if has_think else "❌"}')

Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== PENGUJIAN TAMBAHAN ===

Pertanyaan 1: Apakah PKWT (kontrak kerja waktu tertentu) boleh mensyaratkan masa percobaan kerja?


Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
PKWT adalah kontrak kerja yang biasanya memiliki durasi terbatas, sering kali selama 1 tahun atau kurangnya. Masa percobaan kerja adalah masa awal dari kontrak, dimana pekerja diberikan kesempatan untuk menunjukkan apakah mereka cocok dengan perusahaan. Konsep ini dapat bervariasi antara industri dan negara, tetapi secara umum, tidak ada undang-undang yang memaksa penggunaan masa percobaan...
  >> Format <think>: ✅

Pertanyaan 2: Apa saja hak cuti yang dimiliki pekerja berdasarkan UU Ketenagakerjaan?


Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sebagai model bahasa AI, saya tidak memiliki akses ke informasi terbaru tentang undang-undang atau perubahan yang mungkin telah dilakukan pada UU Ketenagakerjaan. Namun, secara umum, UU Ketenagakerjaan menyediakan beberapa hak cuti bagi pekerja sebagai bentuk perlindungan mereka dari pihak-pihak yang mengganggu kesehatan fisik dan mental mereka. Beberapa hak cuti ini termasuk:

1. Cuti sakit: Peke...
  >> Format <think>: ❌

Pertanyaan 3: Bagaimana ketentuan upah minimum menurut peraturan yang berlaku?


Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Dalam beberapa negara, upah minimum ditetapkan oleh pemerintah atau lembaga independen, biasanya melalui proses regulasi atau undang-undang. Dalam kasus lain, upah minimum dapat ditetapkan oleh badan bisnis atau pengusaha sendiri. Proses penetapan ini sering melibatkan pertimbangan ekonomi seperti tingkat inflasi, pendapatan rata-rata, harga barang dan layanan, serta biaya operasional.
Sem...
  >> Format <think>: ❌

Pertanyaan 4: Apa yang dimaksud dengan PHK dan apa saja alasan yang diperbolehkan?
PHK atau Pemecatan (Terlantar) adalah kejadian ketika pekerja yang didedahkan oleh perusahaan atau lembaga tidak lagi dilakukan kerja mereka karena beberapa alasan seperti kekurangan tenaga kerja, penurunan permintaan produk atau layanan, atau karena bisnis mengalami kesulitan keuangannya.

Alasan yang diperbolehkan untuk pemecatan biasanya termasuk:

1. Kekurangan tenaga kerja: Perusahaan mungkin...
  >> Format <think>: ❌
